In [1]:
import numpy as np
import pandas as pd
pd.options.future.infer_string = False
import pertpy as pt
import scanpy as sc
from scipy.sparse import issparse
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split


def load_and_preprocess(
    n_top_genes: int = 5000,
    target_sum: float = 4000.0,
    min_counts: int = 100,
    min_cells: int = 5,
) -> sc.AnnData:
    adata = pt.data.replogle_2022_k562_essential()
    sc.pp.filter_cells(adata, min_counts=min_counts)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, subset=True)
    perturbed_genes = set(adata.obs["perturbation"].unique()) - {"control"}
    adata.var["highly_variable"] |= adata.var_names.isin(perturbed_genes)
    adata = adata[:, adata.var["highly_variable"]].copy()
    return adata[(adata.obs["nperts"] == 1) | (adata.obs["perturbation"] == "control")].copy()

In [2]:
def make_cell_holdout_split(
    adata: sc.AnnData,
    condition_key: str = "perturbation",
    test_size: float = 0.2,
    min_cells: int = 10,
    random_state: int = 42,
) -> sc.AnnData:
    adata.obs["split"] = "train"
    for pert, group in adata.obs.groupby(condition_key):
        if len(group) < min_cells:
            continue
        _, test_idx = train_test_split(
            group.index.tolist(), test_size=test_size, random_state=random_state
        )
        adata.obs.loc[test_idx, "split"] = "test"
    return adata

In [3]:
class PerturbationModel:
    """Empirical group means with observational/interventional batch marginalisation.

    predict_contrast(T, ctrl) returns (obs, int) contrasts both restricted to
    the common batch support S_{t,0} = S_t ∩ S_0:
      obs = Σ_{b∈S_{t,0}} p̃(b|t)μ(t,b) − Σ_{b∈S_{t,0}} p̃(b|0)μ(0,b)
      int = Σ_{b∈S_{t,0}} q*(b)[μ(t,b)−μ(0,b)],  q*(b) ∝ p(B=b)|_{S_{t,0}}
    """

    def __init__(self, condition_key: str = "perturbation", batch_key: str = "batch"):
        self.ck = condition_key
        self.bk = batch_key

    def fit(self, adata: sc.AnnData) -> "PerturbationModel":
        Y = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
        tb = adata.obs[[self.ck, self.bk]].copy()
        tb["_pos"] = np.arange(len(tb))
        self._means = {
            (t, b): Y[grp["_pos"].values].mean(axis=0)
            for (t, b), grp in tb.groupby([self.ck, self.bk])
        }
        obs = adata.obs
        self._p_B = obs[self.bk].value_counts(normalize=True).sort_index()
        self._p_B_given_T = (
            obs.groupby([self.ck, self.bk]).size()
            .div(obs.groupby(self.ck).size(), level=self.ck)
        )
        return self

    def predict_contrast(self, pert: str, ctrl: str = "control") -> tuple[np.ndarray, np.ndarray]:
        """Common-support (obs, int) contrasts over S_{t,0} = S_t ∩ S_0."""
        S_t = set(self._p_B_given_T.loc[pert].index)
        S_0 = set(self._p_B_given_T.loc[ctrl].index)
        common = sorted(S_t & S_0)
        if not common:
            raise ValueError(f"No common batches between '{pert}' and '{ctrl}'")

        mu_t = np.stack([self._means[(pert, b)] for b in common])
        mu_0 = np.stack([self._means[(ctrl, b)] for b in common])

        w_t = self._p_B_given_T.loc[pert].reindex(common).fillna(0.0).values
        w_t = w_t / w_t.sum()
        w_0 = self._p_B_given_T.loc[ctrl].reindex(common).fillna(0.0).values
        w_0 = w_0 / w_0.sum()
        w_int = self._p_B.reindex(common).fillna(0.0).values
        w_int = w_int / w_int.sum()

        obs_contrast = w_t @ mu_t - w_0 @ mu_0
        int_contrast = w_int @ (mu_t - mu_0)
        return obs_contrast, int_contrast

In [12]:
def evaluate(
    model: PerturbationModel,
    test_adata: sc.AnnData,
    condition_key: str = "perturbation",
    ctrl_label: str = "control",
) -> pd.DataFrame:
    ctrl_mean = np.array(
        test_adata[test_adata.obs[condition_key] == ctrl_label].X.mean(axis=0)
    ).flatten()

    rows = []
    for pert in test_adata.obs[condition_key].unique():
        if pert == ctrl_label:
            continue
        true_cells = test_adata[test_adata.obs[condition_key] == pert]
        if len(true_cells) == 0:
            continue

        naive_contrast = np.array(true_cells.X.mean(axis=0)).flatten() - ctrl_mean
        _, int_contrast = model.predict_contrast(pert, ctrl_label)
        rows.append({
            "obs_int_delta": naive_contrast - int_contrast,
            "n_test_cells":    len(true_cells),
        })

    return pd.DataFrame(rows)

In [13]:
from pathlib import Path

# Kernel CWD may be project root or notebooks/ — check both
_data_dir = next(
    (p for p in [Path("data"), Path("../data")] if p.is_dir()),
    None,
)
if _data_dir is None:
    raise FileNotFoundError(f"Cannot locate data/ directory from CWD: {Path.cwd()}")
PREPROCESSED_PATH = _data_dir / "replogle_k562_preprocessed.h5ad"


def _try_load(path: Path) -> sc.AnnData | None:
    """Load a cached h5ad, returning None (and deleting the file) if invalid."""
    try:
        adata = sc.read_h5ad(path)
        assert "perturbation" in adata.obs.columns
        return adata
    except Exception as e:
        print(f"Cached file invalid ({e}); deleting and reprocessing…")
        path.unlink(missing_ok=True)
        return None


adata = _try_load(PREPROCESSED_PATH) if PREPROCESSED_PATH.exists() else None

if adata is None:
    adata = load_and_preprocess()
    adata.write_h5ad(PREPROCESSED_PATH, compression="gzip")
    print(f"Preprocessed and saved to {PREPROCESSED_PATH}")
else:
    print(f"Loaded preprocessed data from {PREPROCESSED_PATH}")

Loaded preprocessed data from data/replogle_k562_preprocessed.h5ad


In [14]:
# adata = make_cell_holdout_split(adata)

# train_adata = adata[adata.obs["split"] == "train"].copy()
# test_adata  = adata[adata.obs["split"] == "test"].copy()
# # Delete the big data file to save memory
# del adata

In [22]:
# How many unique pertrubations are in the dataset?
adata.obs["perturbation"]

cell_barcode
AAACCCAAGAAATCCA-27       NAF1
AAACCCAAGAACTTCC-31       BUB1
AAACCCAAGAAGCCAC-34       UBL5
AAACCCAAGAATAGTC-43    C9orf16
AAACCCAAGACAGCGT-28      TIMM9
                        ...   
TTTGTTGTCTGTCGTC-45    ATP6V1D
TTTGTTGTCTGTCTCG-27      CNOT3
TTTGTTGTCTGTGCGG-44     METTL3
TTTGTTGTCTTGCAGA-14       RPL5
TTTGTTGTCTTTACAC-25     SEC61B
Name: perturbation, Length: 310385, dtype: category
Categories (2058, object): ['AAAS', 'AAMP', 'AARS', 'AARS2', ..., 'ZRSR2', 'ZW10', 'ZWINT', 'control']

In [15]:
model = PerturbationModel().fit(adata)
results = evaluate(model, adata)

In [16]:
print(results[["obs_int_delta"]].describe())

                                            obs_int_delta
count                                                2057
unique                                               2057
top     [-0.002683357719940436, -0.029746163223773534,...
freq                                                    1


In [20]:
results['obs_int_delta'].mean(), results['obs_int_delta'].std()

TypeError: setting an array element with a sequence.